In [ ]:
# Importing libraries
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from IPython.display import display
from scipy.stats import zscore, normaltest
import scipy.stats as stats
from sklearn import metrics, preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, explained_variance_score, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OrdinalEncoder, OneHotEncoder, PolynomialFeatures
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor, BaggingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from statsmodels.formula import api
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, SelectFromModel, RFE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score
import optuna

# Configure matplotlib
plt.rcParams['figure.figsize'] = [10, 6]

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')


Combination 1: Feature Scaling (Standard) + Cross-Validation (K-Fold) + XGBoost (Default)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
train_data = train_data.drop(columns=['ID'])
test_ids = test_data['ID']
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_data_scaled = scaler.transform(test_data)

# Train XGBoost model with cross-validation
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print("Cross-Validation AUC Scores:", cv_scores)

# Train and evaluate
model.fit(X_train_scaled, y_train)
val_preds = model.predict_proba(X_val_scaled)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = model.predict_proba(test_data_scaled)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination1_FeatureScaling_Standard_XGBoost.csv', index=False)


Combination 2: Feature Reduction (PCA) + Feature Scaling (Min-Max) + Cross-Validation (K-Fold) + XGBoost (Default) + simple imputer

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
train_data = train_data.drop(columns=['ID'])
test_ids = test_data['ID']
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values
imputer = SimpleImputer(strategy='mean')  # You can also use 'median' or 'most_frequent'
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_data_scaled = scaler.transform(test_data)

# PCA for feature reduction
pca = PCA(n_components=0.95)  # Retain 95% variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
test_data_pca = pca.transform(test_data_scaled)

# Train XGBoost model with cross-validation
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
cv_scores = cross_val_score(model, X_train_pca, y_train, cv=5, scoring='roc_auc')
print("Cross-Validation AUC Scores:", cv_scores)

# Train and evaluate
model.fit(X_train_pca, y_train)
val_preds = model.predict_proba(X_val_pca)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = model.predict_proba(test_data_pca)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination2_PCA_MinMax_XGBoost.csv', index=False)



Combination 3: Feature Engineering (One-Hot Encoding, sparse=true) + XGBoost (Default)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Identify categorical columns
categorical_columns = X.select_dtypes(include=['object', 'category']).columns

# Handle missing values
X = X.fillna("Missing")  # Fill missing values in categorical columns
test_data = test_data.fillna("Missing")

# Split data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply One-Hot Encoding with sparse output
encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)
X_train_encoded = encoder.fit_transform(X_train[categorical_columns])
X_val_encoded = encoder.transform(X_val[categorical_columns])
test_data_encoded = encoder.transform(test_data[categorical_columns])

# Concatenate encoded features with numerical columns
numerical_columns = X.select_dtypes(exclude=['object', 'category']).columns
X_train_final = np.hstack((X_train[numerical_columns].values, X_train_encoded.toarray()))
X_val_final = np.hstack((X_val[numerical_columns].values, X_val_encoded.toarray()))
test_data_final = np.hstack((test_data[numerical_columns].values, test_data_encoded.toarray()))

# Train XGBoost model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train_final, y_train)

# Evaluate on validation set
val_preds = model.predict_proba(X_val_final)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = model.predict_proba(test_data_final)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination3_OneHotEncoding_XGBoost.csv', index=False)



Combination 4: Simple Imputer (Median) + Feature Reduction (PCA) + Feature Scaling (Standard) + Cross-Validation (Stratified K-Fold) + XGBoost (Default)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

# Load dataset
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Impute missing values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_data_scaled = scaler.transform(test_data)

# PCA for feature reduction
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_scaled)
test_data_reduced = pca.transform(test_data_scaled)

# Cross-validation and model training
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
strat_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_reduced, y, cv=strat_kfold, scoring='roc_auc')
print("Cross-Validation AUC Scores:", cv_scores)

# Train final model and predict on test set
model.fit(X_reduced, y)
test_preds = model.predict_proba(test_data_reduced)[:, 1]

# Save predictions
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination4_SimpleImputer_Median_PCA_Standard_XGBoost.csv', index=False)




Combination 5: KNN Imputer + Feature Scaling (Standard) + Cross-Validation (K-Fold) + XGBoost (Optimized with Optuna)

In [ ]:
import pandas as pd
import optuna
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, roc_auc_score

# Load dataset
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_data_scaled = scaler.transform(test_data)

# Define objective function for Optuna
def objective(trial):
    # Hyperparameter space
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10, step=0.1),
        "random_state": 42,
    }

    # Cross-validation with KFold
    model = XGBClassifier(**param, use_label_encoder=False, eval_metric="logloss")
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_scaled, y, cv=kfold, scoring="roc_auc")

    return cv_scores.mean()

# Optimize hyperparameters using Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Best parameters
best_params = study.best_params
print("Best Parameters:", best_params)

# Train final model with best parameters
final_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric="logloss")
final_model.fit(X_scaled, y)

# Predict on test set
test_preds = final_model.predict_proba(test_data_scaled)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination5_KNNImputer_Standard_XGBoost_Optuna.csv', index=False)

# Evaluate best model with cross-validation
kfold = KFold(n_splits=2, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X_scaled, y, cv=kfold, scoring="roc_auc")
print("Cross-Validation AUC Scores with Optimized Model:", cv_scores)
print("Mean AUC Score:", cv_scores.mean())


Combination 6: KNN Imputer + Standard Scaling + XGBoost (Optuna) + Feature Reduction (Variance Threshold) + Oversampling (SMOTE)

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import roc_auc_score

# Load dataset
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_data_scaled = scaler.transform(test_data)

# Feature reduction: Variance Threshold
threshold = 0.01  # Keep features with variance above this threshold
var_thresh = VarianceThreshold(threshold=threshold)
X_reduced = var_thresh.fit_transform(X_scaled)
test_data_reduced = var_thresh.transform(test_data_scaled)

# Oversampling using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_reduced, y)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Define Optuna optimization function
def objective(trial):
    # Hyperparameter space
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10, step=0.1),
        "random_state": 42,
    }

    # Cross-validation with KFold
    model = XGBClassifier(**param, use_label_encoder=False, eval_metric="logloss")
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")

    return cv_scores.mean()

# Optimize hyperparameters using Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Best parameters
best_params = study.best_params
print("Best Parameters:", best_params)

# Train final model with best parameters
final_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric="logloss")
final_model.fit(X_train, y_train)

# Evaluate on validation set
val_preds = final_model.predict_proba(X_val)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = final_model.predict_proba(test_data_reduced)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination6_KNNImputer_VarianceThreshold_SMOTE_XGBoost_Optuna.csv', index=False)

# Evaluate final model with cross-validation
cv_scores = cross_val_score(final_model, X_train, y_train, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring="roc_auc")
print("Cross-Validation AUC Scores with Optimized Model:", cv_scores)
print("Mean AUC Score:", cv_scores.mean())


Combination 7: KNN Imputer + Standard Scaling + Feature Reduction (PCA) + XGBoost (Optuna)

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import roc_auc_score

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_data_scaled = scaler.transform(test_data)

# Feature reduction: PCA
pca = PCA(n_components=0.95)  # Retain 95% of variance
X_reduced = pca.fit_transform(X_scaled)
test_data_reduced = pca.transform(test_data_scaled)

# Define Optuna objective function
def objective(trial):
    # Hyperparameter space
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10, step=0.1),
        "random_state": 42,
    }

    # Cross-validation with KFold
    model = XGBClassifier(**param, use_label_encoder=False, eval_metric="logloss")
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_reduced, y, cv=kfold, scoring="roc_auc")

    return cv_scores.mean()

# Optimize hyperparameters using Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Best parameters
best_params = study.best_params
print("Best Parameters:", best_params)

# Train final model with best parameters
final_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric="logloss")
final_model.fit(X_reduced, y)

# Predict on test set
test_preds = final_model.predict_proba(test_data_reduced)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination7_KNNImputer_PCA_Standard_XGBoost_Optuna.csv', index=False)

# Evaluate model with cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X_reduced, y, cv=kfold, scoring="roc_auc")
print("Cross-Validation AUC Scores with Optimized Model:", cv_scores)
print("Mean AUC Score:", cv_scores.mean())


Combination 8: KNN Imputer + SMOTE + XGBoost (Optuna)


In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from xgboost import XGBClassifier
import optuna
from sklearn.metrics import roc_auc_score

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Oversampling using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Define Optuna objective function
def objective(trial):
    # Hyperparameter space
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10, step=0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10, step=0.1),
        "random_state": 42,
    }

    # Cross-validation with KFold
    model = XGBClassifier(**param, use_label_encoder=False, eval_metric="logloss")
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")

    return cv_scores.mean()

# Optimize hyperparameters using Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Best parameters
best_params = study.best_params
print("Best Parameters:", best_params)

# Train final model with best parameters
final_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric="logloss")
final_model.fit(X_train, y_train)

# Evaluate on validation set
val_preds = final_model.predict_proba(X_val)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = final_model.predict_proba(test_data)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination8_KNNImputer_SMOTE_XGBoost_Optuna.csv', index=False)

# Evaluate model with cross-validation
cv_scores = cross_val_score(final_model, X_train, y_train, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring="roc_auc")
print("Cross-Validation AUC Scores with Optimized Model:", cv_scores)
print("Mean AUC Score:", cv_scores.mean())


Combination 9: KNN Imputer + Feature Scaling (Standard) + Cross-Validation (K-Fold) + Voting Ensemble (Optuna)

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import optuna

# Load datasets
train_data = pd.read_csv('fda_trainingset.csv')
test_data = pd.read_csv('fda_testset.csv')

# Drop redundant columns
test_ids = test_data['ID']
train_data = train_data.drop(columns=['ID'])
test_data = test_data.drop(columns=['ID'])

# Split features and target
X = train_data.drop(columns=['Y'])
y = train_data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X = imputer.fit_transform(X)
test_data = imputer.transform(test_data)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_data_scaled = scaler.transform(test_data)

# Train-test split for validation
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define Optuna optimization functions for each model
def optimize_xgboost(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.1),
        "random_state": 42,
    }
    model = XGBClassifier(**param, use_label_encoder=False, eval_metric="logloss")
    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

def optimize_random_forest(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "random_state": 42,
    }
    model = RandomForestClassifier(**param)
    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

def optimize_lightgbm(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", -1, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, step=0.01),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "random_state": 42,
    }
    model = LGBMClassifier(**param)
    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

# Optimize each model with Optuna
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(optimize_xgboost, n_trials=50)
xgb_best_params = study_xgb.best_params

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(optimize_random_forest, n_trials=50)
rf_best_params = study_rf.best_params

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(optimize_lightgbm, n_trials=50)
lgbm_best_params = study_lgbm.best_params

# Initialize models with best parameters
xgb_model = XGBClassifier(**xgb_best_params, use_label_encoder=False, eval_metric="logloss")
rf_model = RandomForestClassifier(**rf_best_params)
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Create voting ensemble
voting_ensemble = VotingClassifier(
    estimators=[("xgb", xgb_model), ("rf", rf_model), ("lgbm", lgbm_model)],
    voting="soft"
)

# Train ensemble model
voting_ensemble.fit(X_train, y_train)

# Evaluate on validation set
val_preds = voting_ensemble.predict_proba(X_val)[:, 1]
print("Validation AUC:", roc_auc_score(y_val, val_preds))

# Predict on test set
test_preds = voting_ensemble.predict_proba(test_data_scaled)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({'ID': test_ids, 'Y': test_preds})
output.to_csv('Combination9_KNNImputer_Standard_VotingEnsemble_Optuna.csv', index=False)

# Evaluate ensemble with cross-validation
cv_scores = cross_val_score(voting_ensemble, X_scaled, y, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring="roc_auc")
print("Cross-Validation AUC Scores for Voting Ensemble:", cv_scores)
print("Mean AUC Score:", cv_scores.mean())


Combination 10: KNN Imputer + Feature Scaling (Standard) + Cross-Validation (K-Fold) + XGBoost (Optuna)

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Drop rows with missing target or features
data.dropna(subset=['Y'], inplace=True)

# Identify categorical features based on provided information
categorical_features = ['X4', 'X5', 'X6', 'X8', 'X10', 'X11']

# Convert categorical features to 'category' type
for col in categorical_features:
    if col in data.columns:
        data[col] = data[col].astype('category')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Define Optuna objective function with K-Fold cross-validation
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'use_label_encoder': False,
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
        'enable_categorical': True
    }

    # K-Fold Cross-Validation
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, valid_idx in kfold.split(X_scaled, y):
        X_train, X_valid = X_scaled[train_idx], X_scaled[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dvalid = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)

        model = xgb.train(
            param,
            dtrain,
            num_boost_round=param['n_estimators'],
            evals=[(dvalid, 'validation')],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        preds = model.predict(dvalid)
        auc = roc_auc_score(y_valid, preds)
        auc_scores.append(auc)

    return sum(auc_scores) / len(auc_scores)

# Hyperparameter tuning with Optuna
print("\nStarting hyperparameter tuning with Optuna...")
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=150)  # Increased number of trials for rigor

# Display the best trial results
print("\nNumber of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial
print("  AUC: {}".format(trial.value))
print("  Best hyperparameters: {}".format(trial.params))

# Train the final model with the best hyperparameters
best_params = trial.params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['use_label_encoder'] = False
best_params['random_state'] = 42
best_params['enable_categorical'] = True
best_params['tree_method'] = 'hist'

n_estimators = best_params['n_estimators']
del best_params['n_estimators']

# Train on the entire dataset
dtrain_full = xgb.DMatrix(X_scaled, label=y, enable_categorical=True)
print("\nTraining the final model on the entire dataset...")
final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=n_estimators
)

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Convert categorical features to 'category' type
for col in categorical_features:
    if col in test_data.columns:
        test_data[col] = test_data[col].astype('category')

# Handle missing values in test set
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_imputed = imputer.transform(X_test)

# Standardize test set
X_test_scaled = scaler.transform(X_test_imputed)
dtest = xgb.DMatrix(X_test_scaled, enable_categorical=True)

# Predict on the test set
print("\nGenerating predictions for the test set...")
y_test_pred = final_model.predict(dtest)

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': y_test_pred
})
predictions.to_csv('Combination10_KNNImputer_Standard_XGBoost_Optuna.csv', index=False)
print("Predictions saved to 'Combination10_KNNImputer_Standard_XGBoost_Optuna.csv'.")


Combination 11: XGBoost with Optuna

In [ ]:
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Define Optuna objective function
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'use_label_encoder': False,
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Train the model
    model = xgb.train(
        param,
        dtrain,
        num_boost_round=param['n_estimators'],
        evals=[(dvalid, 'validation')],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    # Predict on validation set
    preds = model.predict(dvalid)

    # Calculate ROC AUC score
    auc = roc_auc_score(y_valid, preds)
    return auc

# Optimize hyperparameters with Optuna
print("\nStarting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Display best trial results
print("\nBest trial:")
trial = study.best_trial
print(f"AUC: {trial.value}")
print(f"Best parameters: {trial.params}")

# Train final model with best parameters
best_params = trial.params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['use_label_encoder'] = False
best_params['random_state'] = 42
best_params['tree_method'] = 'hist'

n_estimators = best_params['n_estimators']
del best_params['n_estimators']

dtrain_full = xgb.DMatrix(X, label=y)
print("\nTraining the final model with the best parameters...")
final_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=n_estimators
)


In [ ]:
# Load the test dataset
test_data = pd.read_csv('fda_testset.csv')  # Replace with the correct path to your test file

# Prepare test data (drop 'ID' if present)
X_test = test_data.drop(['ID'], axis=1, errors='ignore')

# Create a DMatrix for the test dataset
dtest = xgb.DMatrix(X_test)

# Generate predictions using the final trained model
print("\nGenerating predictions for the test set...")
y_test_pred = final_model.predict(dtest)

# Save predictions to a CSV file
predictions = pd.DataFrame({
    'ID': test_data['ID'],  # Ensure 'ID' exists in the test dataset
    'Predicted_Probability': y_test_pred
})

# Export to CSV
predictions.to_csv('Combination11_XGBoost_Optuna.csv', index=False)
print("Predictions saved to 'Combination11_XGBoost_Optuna.csv'.")



Combination 12: Random Forest with PairwiseDifferenceClassifier


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import roc_auc_score
from itertools import combinations

# Define PairwiseDifferenceClassifier
class PairwiseDifferenceClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        # Generate pairwise differences
        X_diff, y_diff = self._generate_pairwise_data(X, y)
        self.base_estimator.fit(X_diff, y_diff)
        return self

    def predict(self, X):
        return self.base_estimator.predict(X)

    def predict_proba(self, X):
        return self.base_estimator.predict_proba(X)

    def _generate_pairwise_data(self, X, y):
        X_diff = []
        y_diff = []

        # Generate all pairwise combinations of indices
        for (i, j) in combinations(range(len(y)), 2):
            X_diff.append(X[i] - X[j])
            y_diff.append(1 if y[i] > y[j] else 0)

        return np.array(X_diff), np.array(y_diff)

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1).values
y = data['Y'].values

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Initialize Random Forest and wrap it with PairwiseDifferenceClassifier
base_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
pairwise_rf = PairwiseDifferenceClassifier(base_estimator=base_rf)

# Train PairwiseDifferenceClassifier
print("\nTraining PairwiseDifferenceClassifier with Random Forest...")
pairwise_rf.fit(X_train, y_train)

# Evaluate on validation set
val_preds = pairwise_rf.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC: {auc}")

# Cross-validation
print("\nPerforming cross-validation...")
cv_scores = cross_val_score(pairwise_rf, X_train, y_train, cv=5, scoring='roc_auc')
print(f"Cross-Validation AUC Scores: {cv_scores}")
print(f"Mean AUC Score: {cv_scores.mean()}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data
X_test = test_data.drop(['ID'], axis=1, errors='ignore').values

# Predict on test set
test_preds = pairwise_rf.predict_proba(X_test)[:, 1]

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination12_RandomForest_PairwiseDifference.csv', index=False)
print("Predictions saved to 'Combination12_RandomForest_PairwiseDifference.csv'.")


Combination 13: XGBoost with Permutation Feature Importance (PFE)


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train XGBoost model
print("\nTraining XGBoost model...")
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate model on validation set
val_preds = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC: {auc}")

# Calculate Permutation Feature Importance (PFE)
print("\nCalculating Permutation Feature Importance...")
pfi_result = permutation_importance(
    model, X_valid, y_valid, scoring="roc_auc", n_repeats=10, random_state=42
)

# Display PFE results
sorted_idx = np.argsort(pfi_result.importances_mean)[::-1]
print("\nFeature Importance (Permutation):")
for idx in sorted_idx:
    print(f"Feature: {X.columns[idx]}, Importance: {pfi_result.importances_mean[idx]:.4f}")

# Select top features based on PFE
top_features = [X.columns[idx] for idx in sorted_idx[:10]]
print("\nTop Features Selected:", top_features)

# Train a new model using only top features
print("\nTraining XGBoost with Top Features...")
X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]
final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
final_model.fit(X_train_top, y_train)

# Evaluate new model on validation set
val_preds_top = final_model.predict_proba(X_valid_top)[:, 1]
auc_top = roc_auc_score(y_valid, val_preds_top)
print(f"\nValidation AUC with Top Features: {auc_top}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_top = X_test[top_features]

# Predict on test set
test_preds = final_model.predict_proba(X_test_top)[:, 1]

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination13_XGBoost_PFE.csv', index=False)
print("Predictions saved to 'Combination13_XGBoost_PFE.csv'.")


Combination 14: Stacking Ensemble with XGBoost, LightGBM, and CatBoost


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Initialize base learners with default parameters (can be tuned using Optuna or GridSearchCV)
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
lgbm_model = LGBMClassifier(random_state=42)
catboost_model = CatBoostClassifier(verbose=0, random_state=42)

# Define stacking ensemble with Logistic Regression as meta-model
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',  # Use probabilities from base learners
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train the stacking ensemble
print("\nTraining the stacking ensemble model...")
stacking_model.fit(X_train, y_train)

# Evaluate on the validation set
print("\nEvaluating the model on validation set...")
val_preds = stacking_model.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data
X_test = test_data.drop(['ID'], axis=1, errors='ignore')

# Generate predictions for the test set
print("\nGenerating predictions for the test set...")
test_preds = stacking_model.predict_proba(X_test)[:, 1]

# Save predictions to a CSV file
predictions = pd.DataFrame({
    'ID': test_data['ID'],  # Ensure test_data has an 'ID' column
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination14_Stacking_XGB_LGBM_CatBoost.csv', index=False)
print("Predictions saved to 'Combination14_Stacking_XGB_LGBM_CatBoost.csv'.")


Combination 15: Stacking Ensemble (LightGBM, CatBoost, XGBoost) with Hyperparameter Tuning (Optuna)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Define Optuna objective functions for hyperparameter tuning

def tune_xgboost(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'tree_method': 'hist',
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    model = XGBClassifier(**param)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

def tune_lightgbm(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = LGBMClassifier(**param)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

def tune_catboost(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'verbose': 0,
    }
    model = CatBoostClassifier(**param)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

# Tune XGBoost
print("Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgboost, n_trials=50)
xgb_best_params = study_xgb.best_params
print("Best XGBoost parameters:", xgb_best_params)

# Tune LightGBM
print("\nTuning LightGBM...")
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(tune_lightgbm, n_trials=50)
lgbm_best_params = study_lgbm.best_params
print("Best LightGBM parameters:", lgbm_best_params)

# Tune CatBoost
print("\nTuning CatBoost...")
study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(tune_catboost, n_trials=50)
catboost_best_params = study_catboost.best_params
print("Best CatBoost parameters:", catboost_best_params)

# Initialize models with best parameters
xgb_model = XGBClassifier(**xgb_best_params, use_label_encoder=False, eval_metric="logloss", random_state=42)
lgbm_model = LGBMClassifier(**lgbm_best_params, random_state=42)
catboost_model = CatBoostClassifier(**catboost_best_params, random_state=42, verbose=0)

# Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data
X_test = test_data.drop(['ID'], axis=1, errors='ignore')

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],  # Ensure test_data contains 'ID'
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination15_Stacking_XGB_LGBM_CatBoost_Optuna.csv', index=False)
print("Predictions saved to 'Combination15_Stacking_XGB_LGBM_CatBoost_Optuna.csv'.")


Combo 16

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier
import lightgbm as lgb
import xgboost as xgb

# Load the training and test datasets
df_train = pd.read_csv("fda_trainingset.csv")
df_test = pd.read_csv("fda_testset.csv")

# Define features and target variable
X = df_train.drop(columns=['ID', 'Y'])  # Drop 'ID' column as per your requirement
y = df_train['Y']

# Step 1: Scale data before imputing to improve KNN effectiveness
scaler = RobustScaler()
X_scaled_initial = scaler.fit_transform(X)
X_scaled_initial_df = pd.DataFrame(X_scaled_initial, columns=X.columns)

# Step 2: Handle missing values using KNN Imputer with updated parameters
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X_scaled_initial_df)
X_imputed_df = pd.DataFrame(X_imputed, columns=X.columns)

# Step 3: Scale data again after imputation
X_scaled_final = scaler.transform(X_imputed_df)
X_scaled_df = pd.DataFrame(X_scaled_final, columns=X.columns)

# Step 4: Train an XGBoost model to get feature importances
xgb_feature_model = xgb.XGBClassifier(random_state=42, n_jobs=-1)
xgb_feature_model.fit(X_scaled_df, y)

# Get feature importances from XGBoost
feature_importances = xgb_feature_model.feature_importances_
importance_df = pd.DataFrame({'Feature': X_scaled_df.columns, 'Importance': feature_importances})

# Apply threshold for feature selection based on XGBoost importances
threshold = 0.011
selected_features = importance_df[importance_df['Importance'] > threshold]['Feature']

# Reduce the feature set based on the selected features
X_reduced = X_scaled_df[selected_features]

# Step 5: Define and customize models with updated base parameters

# LightGBM model
lgbm_model = lgb.LGBMClassifier(
    boosting_type='gbdt',
    learning_rate=0.02,
    num_leaves=128,
    max_depth=12,
    min_data_in_leaf=20,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    lambda_l1=0.3,
    lambda_l2=0.3,
    n_estimators=1500,
    random_state=42,
    n_jobs=-1
)

# AdaBoost model
adaboost_model = AdaBoostClassifier(
    learning_rate=0.5,
    n_estimators=300,
    random_state=42
)

# XGBoost model
xgb_model = xgb.XGBClassifier(
    booster='gbtree',
    learning_rate=0.015,
    max_depth=10,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.8,
    gamma=0.3,
    reg_alpha=0.3,
    reg_lambda=0.6,
    n_estimators=3000,
    random_state=42,
    n_jobs=-1
)

# Set weights based on model performance (hypothetical)
lgbm_weight = 2
adaboost_weight = 1
xgb_weight = 1.5

# Create a voting classifier with weighted models
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm_model),
    ('adaboost', adaboost_model),
    ('xgb', xgb_model)],
    voting='soft', 
    weights=[lgbm_weight, adaboost_weight, xgb_weight],
    n_jobs=-1
)

# Train the final voting classifier on the reduced training set
voting_clf.fit(X_reduced, y)

# Step 6: Impute and scale the test set data using updated KNN Imputer and RobustScaler
df_test = df_test.drop(columns=['ID'], errors='ignore')  # Drop 'ID' column as required
df_test_scaled_initial = scaler.transform(df_test)
df_test_imputed = knn_imputer.transform(df_test_scaled_initial)
df_test_scaled_final = scaler.transform(df_test_imputed)

# Reduce the test set features based on the selected features
df_test_reduced = pd.DataFrame(df_test_scaled_final, columns=X_scaled_df.columns)[selected_features]

# Predict probabilities on the test set
final_probabilities = voting_clf.predict_proba(df_test_reduced)[:, 1]

# Export predictions to CSV
output = pd.DataFrame({'Predicted_Probability': final_probabilities})
output.to_csv('Combination16_Voting_Classifier.csv', index=False)
print("Predictions saved to 'Combination16_Voting_Classifier.csv'.")


Combination 17: XGBoost + Permutation Feature Importance (PFE) + Optuna Hyperparameter Tuning

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Define Optuna objective function
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'random_state': 42,
        'tree_method': 'hist',
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
    }

    # Train model with hyperparameters
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    val_preds = model.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, val_preds)

# Perform hyperparameter tuning with Optuna
print("\nStarting Optuna hyperparameter tuning...")
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# Display the best parameters and AUC
print("\nBest trial:")
print(f"Validation AUC: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Train model with the best hyperparameters
best_params = study.best_params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['use_label_encoder'] = False
best_params['random_state'] = 42
best_params['tree_method'] = 'hist'

print("\nTraining XGBoost model with best parameters...")
model = xgb.XGBClassifier(**best_params)
model.fit(X_train, y_train)

# Evaluate model on validation set
val_preds = model.predict_proba(X_valid)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC with best parameters: {auc:.4f}")

# Calculate Permutation Feature Importance (PFE)
print("\nCalculating Permutation Feature Importance...")
pfi_result = permutation_importance(
    model, X_valid, y_valid, scoring="roc_auc", n_repeats=10, random_state=42
)

# Display PFE results
sorted_idx = np.argsort(pfi_result.importances_mean)[::-1]
print("\nFeature Importance (Permutation):")
for idx in sorted_idx:
    print(f"Feature: {X.columns[idx]}, Importance: {pfi_result.importances_mean[idx]:.4f}")

# Select top features based on PFE
top_features = [X.columns[idx] for idx in sorted_idx[:10]]
print("\nTop Features Selected:", top_features)

# Train a new model using only top features
print("\nTraining XGBoost with Top Features...")
X_train_top = X_train[top_features]
X_valid_top = X_valid[top_features]
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_top, y_train)

# Evaluate new model on validation set
val_preds_top = final_model.predict_proba(X_valid_top)[:, 1]
auc_top = roc_auc_score(y_valid, val_preds_top)
print(f"\nValidation AUC with Top Features: {auc_top:.4f}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_top = X_test[top_features]

# Predict on test set
test_preds = final_model.predict_proba(X_test_top)[:, 1]

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination17_XGBoost_Optuna_PFE.csv', index=False)
print("Predictions saved to 'Combination17_XGBoost_Optuna_PFE.csv'.")


Combination 18: XGBoost with Optuna Tuning + LightGBM Feature Selection

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: LightGBM for Feature Selection
print("\nTraining LightGBM for Feature Selection...")
lgbm_model = lgb.LGBMClassifier(
    boosting_type='gbdt',
    learning_rate=0.1,
    num_leaves=31,
    n_estimators=100,
    random_state=42
)
lgbm_model.fit(X_train, y_train)

# Get feature importances from LightGBM
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgbm_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., top 10% of features based on importance)
top_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {top_features}")

# Reduce dataset to top features
X_train_reduced = X_train[top_features]
X_valid_reduced = X_valid[top_features]

# Step 2: Optuna for XGBoost Hyperparameter Tuning
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'tree_method': 'hist',
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train_reduced, y_train)
    val_preds = model.predict_proba(X_valid_reduced)[:, 1]
    return roc_auc_score(y_valid, val_preds)

# Perform hyperparameter tuning with Optuna
print("\nStarting Optuna hyperparameter tuning for XGBoost...")
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# Display the best parameters and AUC
print("\nBest trial:")
print(f"Validation AUC: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Step 3: Train Final XGBoost Model with Best Parameters
best_params = study.best_params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['use_label_encoder'] = False
best_params['random_state'] = 42
best_params['tree_method'] = 'hist'

print("\nTraining XGBoost model with best parameters...")
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
val_preds = final_model.predict_proba(X_valid_reduced)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC with top features: {auc:.4f}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Prepare test data with selected top features
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_reduced = X_test[top_features]

# Predict on test set
test_preds = final_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],  # Retain test IDs
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination18_XGBoost_LGBM_FS.csv', index=False)
print("Predictions saved to 'Combination18_XGBoost_LGBM_FS.csv'.")


Combination 19: XGBoost with Optuna Tuning + Random Forest Feature Selection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using KNN Imputer
print("\nHandling missing values using KNN Imputer...")
knn_imputer = KNNImputer(n_neighbors=5, weights="uniform")
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: Random Forest for Feature Selection
print("\nTraining Random Forest for Feature Selection...")
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Get feature importances from Random Forest
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., top 10% of features based on importance)
top_features = feature_importances[feature_importances['Importance'] > 0.01]['Feature'].tolist()
print(f"\nSelected Top Features: {top_features}")

# Reduce datasets to top features
X_train_reduced = X_train[top_features]
X_valid_reduced = X_valid[top_features]

# Step 2: Optuna for XGBoost Hyperparameter Tuning
def objective(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'use_label_encoder': False,
        'tree_method': 'hist',
        'random_state': 42,
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 10.0),
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train_reduced, y_train)
    val_preds = model.predict_proba(X_valid_reduced)[:, 1]
    return roc_auc_score(y_valid, val_preds)

# Perform hyperparameter tuning with Optuna
print("\nStarting Optuna hyperparameter tuning for XGBoost...")
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# Display the best parameters and AUC
print("\nBest trial:")
print(f"Validation AUC: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Step 3: Train Final XGBoost Model with Best Parameters
best_params = study.best_params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = 'auc'
best_params['use_label_encoder'] = False
best_params['random_state'] = 42
best_params['tree_method'] = 'hist'

print("\nTraining XGBoost model with best parameters...")
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
val_preds = final_model.predict_proba(X_valid_reduced)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC with top features: {auc:.4f}")

# Load test dataset
test_data = pd.read_csv('fda_testset.csv')

# Handle missing values in the test dataset using KNN Imputer
test_data_imputed = knn_imputer.transform(test_data.drop(['ID'], axis=1, errors='ignore'))

# Reduce test set features based on the selected features
X_test_reduced = pd.DataFrame(test_data_imputed, columns=X.columns)[top_features]

# Predict on test set
test_preds = final_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to a CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination19_XGBoost_RF_FS_KNN.csv', index=False)
print("Predictions saved to 'Combination19_XGBoost_RF_FS_KNN.csv'.")



Combination 20: Voting Classifier (XGBoost, LightGBM, AdaBoost) with Feature Selection (XGBoost)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier
import lightgbm as lgb
import xgboost as xgb

# Load the training and test datasets
df_train = pd.read_csv("fda_trainingset.csv")
df_test = pd.read_csv("fda_testset.csv")

# Define features and target variable
X = df_train.drop(columns=['ID', 'Y'])  # Drop 'ID' column
y = df_train['Y']

# Step 1: Train an XGBoost model to get feature importances
xgb_feature_model = xgb.XGBClassifier(random_state=42, n_jobs=-1)
xgb_feature_model.fit(X, y)

# Get feature importances from XGBoost
feature_importances = xgb_feature_model.feature_importances_
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})

# Apply threshold for feature selection based on XGBoost importances
threshold = 0.011  # Features with importance greater than the threshold
selected_features = importance_df[importance_df['Importance'] > threshold]['Feature']

# Reduce the feature set based on the selected features
X_reduced = X[selected_features]

# Step 2: Define and customize models with updated base parameters

# LightGBM model
lgbm_model = lgb.LGBMClassifier(
    boosting_type='gbdt',
    learning_rate=0.02,
    num_leaves=128,
    max_depth=12,
    min_data_in_leaf=20,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=5,
    lambda_l1=0.3,
    lambda_l2=0.3,
    n_estimators=1500,
    random_state=42,
    n_jobs=-1
)

# AdaBoost model
adaboost_model = AdaBoostClassifier(
    learning_rate=0.5,
    n_estimators=300,
    random_state=42
)

# XGBoost model
xgb_model = xgb.XGBClassifier(
    booster='gbtree',
    learning_rate=0.015,
    max_depth=10,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.8,
    gamma=0.3,
    reg_alpha=0.3,
    reg_lambda=0.6,
    n_estimators=3000,
    random_state=42,
    n_jobs=-1
)

# Set weights based on model performance (hypothetical)
lgbm_weight = 2
adaboost_weight = 1
xgb_weight = 1.5

# Create a voting classifier with weighted models
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm_model),
    ('adaboost', adaboost_model),
    ('xgb', xgb_model)],
    voting='soft', 
    weights=[lgbm_weight, adaboost_weight, xgb_weight],
    n_jobs=-1
)

# Train the final voting classifier on the reduced training set
voting_clf.fit(X_reduced, y)

# Step 3: Prepare test set and reduce features based on selected features
test_ids = df_test['ID']  # Retain IDs for output
X_test = df_test.drop(columns=['ID'], errors='ignore')
X_test_reduced = X_test[selected_features]

# Predict probabilities on the test set
final_probabilities = voting_clf.predict_proba(X_test_reduced)[:, 1]

# Export predictions to CSV
output = pd.DataFrame({
    'ID': test_ids,
    'Predicted_Probability': final_probabilities
})
output.to_csv('Combination20_Voting_Classifier.csv', index=False)
print("Predictions saved to 'Combination20_Voting_Classifier.csv'.")


combo 21

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: LightGBM for Feature Selection
print("\nTraining LightGBM for Feature Selection...")
lgbm_fs_model = LGBMClassifier(
    boosting_type='gbdt',
    learning_rate=0.1,
    num_leaves=31,
    n_estimators=100,
    random_state=42
)
lgbm_fs_model.fit(X_train, y_train)

# Get feature importances from LightGBM
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgbm_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., top 10% of features based on importance)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Step 2: Initialize models with provided parameters for XGBoost and LightGBM
xgb_best_params = {
    'max_depth': 3,
    'learning_rate': 0.035936064805401076,
    'n_estimators': 753,
    'gamma': 2.041403475874556,
    'min_child_weight': 7,
    'subsample': 0.9389471973168744,
    'colsample_bytree': 0.9469938285171243,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}
xgb_model = XGBClassifier(**xgb_best_params)

lgbm_best_params = {
    'max_depth': 4,
    'learning_rate': 0.03021357938385291,
    'n_estimators': 429,
    'num_leaves': 101,
    'subsample': 0.9281930664068806,
    'colsample_bytree': 0.6893797899138006,
    'min_child_weight': 5,
    'random_state': 42
}
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Step 3: Define Optuna objective for CatBoost hyperparameter tuning
def tune_catboost(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'random_state': 42,
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'verbose': 0,
    }
    model = CatBoostClassifier(**param)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_reduced, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

# Step 4: Tune CatBoost using Optuna
print("\nTuning CatBoost...")
study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(tune_catboost, n_trials=50)
catboost_best_params = study_catboost.best_params
print("Best CatBoost parameters:", catboost_best_params)

# Initialize CatBoost with best parameters
catboost_model = CatBoostClassifier(**catboost_best_params, random_state=42, verbose=0)

# Step 5: Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Step 6: Load and prepare test data
test_data = pd.read_csv('fda_testset.csv')
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_reduced = X_test[selected_features]

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination21_Stacking_XGB_LGBM_CatBoost_LGBM_FS.csv', index=False)
print("Predictions saved to 'Combination21_Stacking_XGB_LGBM_CatBoost_LGBM_FS.csv'.")


Combination 22: Voting Classifier with Optuna-Tuned Parameters + LightGBM for Feature Selection

Best XGBoost parameters: {'max_depth': 14, 'learning_rate': 0.017777886772191234, 'n_estimators': 807, 'gamma': 3.652713784991863, 'min_child_weight': 7, 'subsample': 0.6060276329611571, 'colsample_bytree': 0.8336220511759428}
Best LightGBM parameters: {'max_depth': 14, 'learning_rate': 0.020261475950029136, 'n_estimators': 320, 'num_leaves': 89, 'subsample': 0.8313660657744985, 'colsample_bytree': 0.62699569878381, 'min_child_weight': 5}

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import lightgbm as lgb
import xgboost as xgb
import optuna

# Load the training and test datasets
df_train = pd.read_csv("fda_trainingset.csv")
df_test = pd.read_csv("fda_testset.csv")

# Define features and target variable
X = df_train.drop(columns=['ID', 'Y'])  # Drop 'ID' column
y = df_train['Y']

# Step 1: Handle missing values using KNN Imputer
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Step 2: Use LightGBM for Feature Selection
print("\nTraining LightGBM for Feature Selection...")
lgb_fs_model = lgb.LGBMClassifier(random_state=42, n_estimators=100)
lgb_fs_model.fit(X, y)

# Get feature importances
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., features with importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce the feature set
X_reduced = X[selected_features]

# Initialize AdaBoost model with fixed parameters
adaboost_model = AdaBoostClassifier(
    learning_rate=0.5,
    n_estimators=300,
    random_state=42
)

# Initialize pre-tuned models
xgb_best_params = {
    'max_depth': 14,
    'learning_rate': 0.017777886772191234,
    'n_estimators': 807,
    'gamma': 3.652713784991863,
    'min_child_weight': 7,
    'subsample': 0.6060276329611571,
    'colsample_bytree': 0.8336220511759428,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}
xgb_model = xgb.XGBClassifier(**xgb_best_params)

lgbm_best_params = {
    'max_depth': 14,
    'learning_rate': 0.020261475950029136,
    'n_estimators': 320,
    'num_leaves': 89,
    'subsample': 0.8313660657744985,
    'colsample_bytree': 0.62699569878381,
    'min_child_weight': 5,
    'random_state': 42
}
lgbm_model = lgb.LGBMClassifier(**lgbm_best_params)

# Step 3: Create a voting classifier
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm_model),
    ('adaboost', adaboost_model),
    ('xgb', xgb_model)],
    voting='soft', 
    weights=[2, 1, 1.5],
    n_jobs=-1
)

# Train the voting classifier
voting_clf.fit(X_reduced, y)

# Step 4: Prepare the test data
X_test = df_test.drop(columns=['ID'], errors='ignore')
X_test_imputed = knn_imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Predict probabilities on the test set
final_probabilities = voting_clf.predict_proba(X_test_reduced)[:, 1]

# Export predictions to CSV
output = pd.DataFrame({
    'ID': df_test['ID'],
    'Predicted_Probability': final_probabilities
})
output.to_csv('Combination22_Voting_Classifier_LGBM_FS_Preset_XGB_LGBM.csv', index=False)
print("Predictions saved to 'Combination22_Voting_Classifier_LGBM_FS_Preset_XGB_LGBM.csv'.")


Combination 23: XGBoost with Optuna Tuning + XGBoost for Feature Selection


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler

# Load dataset
data = pd.read_csv("fda_trainingset.csv")

# Separate features and target variable
X = data.drop(columns=["ID", "Y"])  # Drop 'ID' column
y = data["Y"]

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
feature_selection_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
feature_selection_model.fit(X_train, y_train)

# Get feature importances
feature_importances = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_selection_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

# Select top features based on importance (e.g., importance > 0)
selected_features = feature_importances[feature_importances["Importance"] > 0]["Feature"].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Step 2: Define Optuna objective function for hyperparameter tuning
def objective(trial):
    param = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "use_label_encoder": False,
        "random_state": 42,
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-5, 10.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-5, 10.0),
    }

    # Train model with current parameters
    model = xgb.XGBClassifier(**param)
    model.fit(X_train_reduced, y_train)
    val_preds = model.predict_proba(X_valid_reduced)[:, 1]
    return roc_auc_score(y_valid, val_preds)

# Step 3: Perform hyperparameter tuning with Optuna
print("\nStarting Optuna hyperparameter tuning...")
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# Display the best parameters and AUC
print("\nBest trial:")
print(f"Validation AUC: {study.best_value:.4f}")
print(f"Best parameters: {study.best_params}")

# Step 4: Train final XGBoost model with best parameters
best_params = study.best_params
best_params["objective"] = "binary:logistic"
best_params["eval_metric"] = "auc"
best_params["use_label_encoder"] = False
best_params["random_state"] = 42
best_params["tree_method"] = "hist"

print("\nTraining XGBoost model with best parameters...")
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_reduced, y_train)

# Evaluate final model on validation set
val_preds = final_model.predict_proba(X_valid_reduced)[:, 1]
auc = roc_auc_score(y_valid, val_preds)
print(f"\nValidation AUC with top features: {auc:.4f}")

# Step 5: Load and prepare test dataset
test_data = pd.read_csv("fda_testset.csv")
X_test = test_data.drop(columns=["ID"], errors="ignore")
X_test_reduced = X_test[selected_features]

# Predict probabilities on the test set
print("\nGenerating predictions for test set...")
test_preds = final_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
output = pd.DataFrame({
    "ID": test_data["ID"],
    "Predicted_Probability": test_preds
})
output.to_csv("Combination23_XGBoost_XGB_FS_Optuna.csv", index=False)
print("Predictions saved to 'Combination23_XGBoost_XGB_FS_Optuna.csv'.")


Combination 24: Stacking Ensemble of LightGBM, AdaBoost, XGBoost with Optuna Tuning and XGBoost Feature Selection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X_train, y_train)

# Get feature importances from XGBoost
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Initialize XGBoost and LightGBM with provided parameters
xgb_best_params = {
    'max_depth': 15,
    'learning_rate': 0.01595993302780678,
    'n_estimators': 816,
    'gamma': 1.7987640404534366,
    'min_child_weight': 5,
    'subsample': 0.8538477965918823,
    'colsample_bytree': 0.8287067372905088,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}

lgbm_best_params = {
    'max_depth': 4,
    'learning_rate': 0.02013764708811443,
    'n_estimators': 963,
    'num_leaves': 65,
    'subsample': 0.8772422328969501,
    'colsample_bytree': 0.7454077790665029,
    'min_child_weight': 3,
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_best_params)
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Step 2: Define Optuna objective for AdaBoost hyperparameter tuning
def tune_adaboost(trial):
    try:
        # Trial-defined parameters for AdaBoost
        adaboost_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        }
        adaboost_model = AdaBoostClassifier(**adaboost_params, random_state=42)

        # Define the stacking ensemble
        stacking_model = StackingClassifier(
            estimators=[
                ('xgb', xgb_model),
                ('lgbm', lgbm_model),
                ('adaboost', adaboost_model)
            ],
            final_estimator=LogisticRegression(random_state=42),
            stack_method='predict_proba',
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        )

        # Evaluate the stacking ensemble using cross-validation
        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(stacking_model, X_train_reduced, y_train, cv=kfold, scoring="roc_auc")

        return scores.mean()
    except Exception as e:
        print(f"Trial failed with exception: {e}")
        return 0.0  # Return 0 for failed trials

# Step 3: Tune AdaBoost using Optuna
print("\nTuning AdaBoost...")
study_adaboost = optuna.create_study(direction='maximize')
study_adaboost.optimize(tune_adaboost, n_trials=50)
adaboost_best_params = study_adaboost.best_params
print("Best AdaBoost parameters:", adaboost_best_params)

# Initialize AdaBoost with the best parameters
adaboost_model = AdaBoostClassifier(**adaboost_best_params, random_state=42)

# Step 4: Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('adaboost', adaboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Step 5: Load and prepare test data
test_data = pd.read_csv('fda_testset.csv')
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_imputed = imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination24_Stacking_XGB_LGBM_Adaboost_XGB_FS.csv', index=False)
print("Predictions saved to 'Combination24_Stacking_XGB_LGBM_Adaboost_XGB_FS.csv'.")





In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X_train, y_train)

# Get feature importances from XGBoost
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Step 2: Define Optuna objective for AdaBoost hyperparameter tuning
def tune_adaboost(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 1.0),
    }
    model = AdaBoostClassifier(**param, random_state=42)
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_reduced, y_train, cv=kfold, scoring="roc_auc")
    return scores.mean()

# Step 3: Tune AdaBoost using Optuna
print("\nTuning AdaBoost with Optuna...")
study_adaboost = optuna.create_study(direction='maximize')
study_adaboost.optimize(tune_adaboost, n_trials=25)  # Keeping trials small
adaboost_best_params = study_adaboost.best_params
print("Best AdaBoost parameters:", adaboost_best_params)

# Step 4: Train and evaluate the final AdaBoost model
adaboost_model = AdaBoostClassifier(**adaboost_best_params, random_state=42)
adaboost_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = adaboost_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X_train, y_train)

# Get feature importances from XGBoost
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Initialize XGBoost and LightGBM with provided parameters
xgb_best_params = {
    'max_depth': 15,
    'learning_rate': 0.01595993302780678,
    'n_estimators': 816,
    'gamma': 1.7987640404534366,
    'min_child_weight': 5,
    'subsample': 0.8538477965918823,
    'colsample_bytree': 0.8287067372905088,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}

lgbm_best_params = {
    'max_depth': 4,
    'learning_rate': 0.02013764708811443,
    'n_estimators': 963,
    'num_leaves': 65,
    'subsample': 0.8772422328969501,
    'colsample_bytree': 0.7454077790665029,
    'min_child_weight': 3,
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_best_params)
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Step 2: Define Optuna objective for CatBoost hyperparameter tuning
def tune_catboost(trial):
    try:
        # Trial-defined parameters for CatBoost
        catboost_params = {
            'iterations': trial.suggest_int('iterations', 50, 300),
            'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-3, 10.0),
            'random_state': 42,
            'verbose': 0
        }
        catboost_model = CatBoostClassifier(**catboost_params)

        # Define the stacking ensemble
        stacking_model = StackingClassifier(
            estimators=[
                ('xgb', xgb_model),
                ('lgbm', lgbm_model),
                ('catboost', catboost_model)
            ],
            final_estimator=LogisticRegression(random_state=42),
            stack_method='predict_proba',
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        )

        # Evaluate the stacking ensemble using cross-validation
        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(stacking_model, X_train_reduced, y_train, cv=kfold, scoring="roc_auc")

        return scores.mean()
    except Exception as e:
        print(f"Trial failed with exception: {e}")
        return 0.0  # Return 0 for failed trials

# Step 3: Tune CatBoost using Optuna
print("\nTuning CatBoost...")
study_catboost = optuna.create_study(direction='maximize')
study_catboost.optimize(tune_catboost, n_trials=20)  # Reducing trials for faster execution
catboost_best_params = study_catboost.best_params
print("Best CatBoost parameters:", catboost_best_params)

# Initialize CatBoost with the best parameters
catboost_model = CatBoostClassifier(**catboost_best_params, random_state=42, verbose=0)

# Step 4: Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Step 5: Load and prepare test data
test_data = pd.read_csv('fda_testset.csv')
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_imputed = imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination24_Stacking_XGB_LGBM_Catboost_XGB_FS.csv', index=False)
print("Predictions saved to 'Combination24_Stacking_XGB_LGBM_Catboost_XGB_FS.csv'.")


Combination 25

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import lightgbm as lgb
import xgboost as xgb
import optuna

# Load the training and test datasets
df_train = pd.read_csv("fda_trainingset.csv")
df_test = pd.read_csv("fda_testset.csv")

# Define features and target variable
X = df_train.drop(columns=['ID', 'Y'])  # Drop 'ID' column
y = df_train['Y']

# Step 1: Handle missing values using KNN Imputer
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Step 2: Use XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X, y)

# Get feature importances
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., features with importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce the feature set
X_reduced = X[selected_features]

# Step 3: Define Optuna objective function for AdaBoost hyperparameter tuning
def tune_adaboost(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 1.0),
    }
    model = AdaBoostClassifier(**param, random_state=42)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_reduced, y, cv=kfold, scoring="roc_auc")
    return scores.mean()

# Step 4: Tune AdaBoost hyperparameters using Optuna
print("\nTuning AdaBoost...")
study_adaboost = optuna.create_study(direction='maximize')
study_adaboost.optimize(tune_adaboost, n_trials=50)
adaboost_best_params = study_adaboost.best_params
print("Best AdaBoost parameters:", adaboost_best_params)

# Initialize models with best parameters
xgb_best_params = {
    'max_depth': 14,
    'learning_rate': 0.017777886772191234,
    'n_estimators': 807,
    'gamma': 3.652713784991863,
    'min_child_weight': 7,
    'subsample': 0.6060276329611571,
    'colsample_bytree': 0.8336220511759428,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}
xgb_model = xgb.XGBClassifier(**xgb_best_params)

lgbm_best_params = {
    'max_depth': 14,
    'learning_rate': 0.020261475950029136,
    'n_estimators': 320,
    'num_leaves': 89,
    'subsample': 0.8313660657744985,
    'colsample_bytree': 0.62699569878381,
    'min_child_weight': 5,
    'random_state': 42
}
lgbm_model = lgb.LGBMClassifier(**lgbm_best_params)

adaboost_model = AdaBoostClassifier(**adaboost_best_params, random_state=42)

# Step 5: Create a voting classifier
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm_model),
    ('adaboost', adaboost_model),
    ('xgb', xgb_model)],
    voting='soft', 
    weights=[2, 1, 1.5],
    n_jobs=-1
)

# Train the voting classifier
voting_clf.fit(X_reduced, y)

# Step 6: Prepare the test data
X_test = df_test.drop(columns=['ID'], errors='ignore')
X_test_imputed = knn_imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Predict probabilities on the test set
final_probabilities = voting_clf.predict_proba(X_test_reduced)[:, 1]

# Export predictions to CSV
output = pd.DataFrame({
    'ID': df_test['ID'],
    'Predicted_Probability': final_probabilities
})
output.to_csv('Combination25_Voting_Classifier_XGB_FS_Preset_XGB_LGBM.csv', index=False)
print("Predictions saved to 'Combination25_Voting_Classifier_XGB_FS_Preset_XGB_LGBM.csv'.")


26

In [ ]:
#26
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using SimpleImputer
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X_train, y_train)

# Get feature importances from XGBoost
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Initialize XGBoost and LightGBM with provided parameters
xgb_best_params = {
    'max_depth': 15,
    'learning_rate': 0.01595993302780678,
    'n_estimators': 816,
    'gamma': 1.7987640404534366,
    'min_child_weight': 5,
    'subsample': 0.8538477965918823,
    'colsample_bytree': 0.8287067372905088,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}

lgbm_best_params = {
    'max_depth': 4,
    'learning_rate': 0.02013764708811443,
    'n_estimators': 963,
    'num_leaves': 65,
    'subsample': 0.8772422328969501,
    'colsample_bytree': 0.7454077790665029,
    'min_child_weight': 3,
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_best_params)
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Step 2: Define Optuna objective for AdaBoost hyperparameter tuning
def tune_adaboost(trial):
    try:
        # Trial-defined parameters for AdaBoost
        adaboost_params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300),
            'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        }
        adaboost_model = AdaBoostClassifier(**adaboost_params, random_state=42)

        # Define the stacking ensemble
        stacking_model = StackingClassifier(
            estimators=[
                ('xgb', xgb_model),
                ('lgbm', lgbm_model),
                ('adaboost', adaboost_model)
            ],
            final_estimator=LogisticRegression(random_state=42),
            stack_method='predict_proba',
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        )

        # Evaluate the stacking ensemble using cross-validation
        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(stacking_model, X_train_reduced, y_train, cv=kfold, scoring="roc_auc")

        return scores.mean()
    except Exception as e:
        print(f"Trial failed with exception: {e}")
        return 0.0  # Return 0 for failed trials

# Step 3: Tune AdaBoost using Optuna
print("\nTuning AdaBoost...")
study_adaboost = optuna.create_study(direction='maximize')
study_adaboost.optimize(tune_adaboost, n_trials=50)
adaboost_best_params = study_adaboost.best_params
print("Best AdaBoost parameters:", adaboost_best_params)

# Initialize AdaBoost with the best parameters
adaboost_model = AdaBoostClassifier(**adaboost_best_params, random_state=42)

# Step 4: Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('adaboost', adaboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Step 5: Load and prepare test data
test_data = pd.read_csv('fda_testset.csv')
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_imputed = imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination26_Stacking_XGB_LGBM_Adaboost_XGB_FS.csv', index=False)
print("Predictions saved to 'Combination26_Stacking_XGB_LGBM_Adaboost_XGB_FS.csv'.")

28

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.ensemble import StackingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Load dataset
data = pd.read_csv('fda_trainingset.csv')

# Separate features and target variable
X = data.drop(['ID', 'Y'], axis=1)
y = data['Y']

# Handle missing values using KNNImputer
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Split data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 1: XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X_train, y_train)

# Get feature importances from XGBoost
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce training and validation datasets to selected features
X_train_reduced = X_train[selected_features]
X_valid_reduced = X_valid[selected_features]

# Initialize XGBoost and LightGBM with provided parameters
xgb_best_params = {
    'max_depth': 15,
    'learning_rate': 0.01595993302780678,
    'n_estimators': 816,
    'gamma': 1.7987640404534366,
    'min_child_weight': 5,
    'subsample': 0.8538477965918823,
    'colsample_bytree': 0.8287067372905088,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}

lgbm_best_params = {
    'max_depth': 4,
    'learning_rate': 0.02013764708811443,
    'n_estimators': 963,
    'num_leaves': 65,
    'subsample': 0.8772422328969501,
    'colsample_bytree': 0.7454077790665029,
    'min_child_weight': 3,
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_best_params)
lgbm_model = LGBMClassifier(**lgbm_best_params)

# Step 2: Initialize AdaBoost with fixed parameters
adaboost_model = AdaBoostClassifier(
    learning_rate=0.5,
    n_estimators=300,
    random_state=42
)

# Step 3: Create stacking ensemble
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('adaboost', adaboost_model)
    ],
    final_estimator=LogisticRegression(random_state=42),
    stack_method='predict_proba',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
)

# Train stacking ensemble
print("\nTraining stacking ensemble...")
stacking_model.fit(X_train_reduced, y_train)

# Evaluate on validation set
print("\nEvaluating on validation set...")
val_preds = stacking_model.predict_proba(X_valid_reduced)[:, 1]
auc_score = roc_auc_score(y_valid, val_preds)
print(f"Validation AUC: {auc_score:.4f}")

# Step 4: Load and prepare test data
test_data = pd.read_csv('fda_testset.csv')
X_test = test_data.drop(['ID'], axis=1, errors='ignore')
X_test_imputed = knn_imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Generate predictions on test set
print("\nGenerating predictions for test set...")
test_preds = stacking_model.predict_proba(X_test_reduced)[:, 1]

# Save predictions to CSV
predictions = pd.DataFrame({
    'ID': test_data['ID'],
    'Predicted_Probability': test_preds
})
predictions.to_csv('Combination28_Stacking_XGB_LGBM_AdaBoost_XGB_FS.csv', index=False)
print("Predictions saved to 'Combination28_Stacking_XGB_LGBM_AdaBoost_XGB_FS.csv'.")


29

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import lightgbm as lgb
import xgboost as xgb

# Load the training and test datasets
df_train = pd.read_csv("fda_trainingset.csv")
df_test = pd.read_csv("fda_testset.csv")

# Define features and target variable
X = df_train.drop(columns=['ID', 'Y'])  # Drop 'ID' column
y = df_train['Y']

# Step 1: Handle missing values using KNN Imputer
knn_imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Step 2: Use XGBoost for Feature Selection
print("\nTraining XGBoost for Feature Selection...")
xgb_fs_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    use_label_encoder=False,
    random_state=42
)
xgb_fs_model.fit(X, y)

# Get feature importances
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_fs_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Select top features (e.g., features with importance > 0)
selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
print(f"\nSelected Top Features: {selected_features}")

# Reduce the feature set
X_reduced = X[selected_features]

# Initialize models with predefined parameters
xgb_best_params = {
    'max_depth': 14,
    'learning_rate': 0.017777886772191234,
    'n_estimators': 807,
    'gamma': 3.652713784991863,
    'min_child_weight': 7,
    'subsample': 0.6060276329611571,
    'colsample_bytree': 0.8336220511759428,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'use_label_encoder': False,
    'random_state': 42
}
xgb_model = xgb.XGBClassifier(**xgb_best_params)

lgbm_best_params = {
    'max_depth': 14,
    'learning_rate': 0.020261475950029136,
    'n_estimators': 320,
    'num_leaves': 89,
    'subsample': 0.8313660657744985,
    'colsample_bytree': 0.62699569878381,
    'min_child_weight': 5,
    'random_state': 42
}
lgbm_model = lgb.LGBMClassifier(**lgbm_best_params)

# Define AdaBoost model with given parameters
adaboost_model = AdaBoostClassifier(
    learning_rate=0.5,
    n_estimators=300,
    random_state=42
)

# Step 3: Create a voting classifier
voting_clf = VotingClassifier(estimators=[
    ('lgbm', lgbm_model),
    ('adaboost', adaboost_model),
    ('xgb', xgb_model)],
    voting='soft', 
    weights=[2, 1, 1.5],
    n_jobs=-1
)

# Train the voting classifier
voting_clf.fit(X_reduced, y)

# Step 4: Prepare the test data
X_test = df_test.drop(columns=['ID'], errors='ignore')
X_test_imputed = knn_imputer.transform(X_test)
X_test_reduced = pd.DataFrame(X_test_imputed, columns=X.columns)[selected_features]

# Predict probabilities on the test set
final_probabilities = voting_clf.predict_proba(X_test_reduced)[:, 1]

# Export predictions to CSV
output = pd.DataFrame({
    'ID': df_test['ID'],
    'Predicted_Probability': final_probabilities
})
output.to_csv('Combination29_Voting_Classifier_XGB_FS_Preset_XGB_LGBM_AdaBoost.csv', index=False)
print("Predictions saved to 'Combination29_Voting_Classifier_XGB_FS_Preset_XGB_LGBM_AdaBoost.csv'.")


### EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split

# Load the dataset
file_name = "fda_testset.csv"
df = pd.read_csv(file_name)

# Display basic info
print("Basic Information:")
print(df.info())

# Summary statistics
print("\nSummary Statistics:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check unique values in each column
print("\nUnique Values per Column:")
print(df.nunique())

# Check class imbalance if there's a target column
if 'target' in df.columns:  # Replace 'target' with your actual target column name
    print("\nTarget Class Distribution:")
    print(df['target'].value_counts(normalize=True))
    df['target'].value_counts().plot(kind='bar', title='Target Class Distribution')
    plt.show()

# Correlation heatmap
print("\nCorrelation Heatmap:")
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

# Check for outliers using boxplots
print("\nBoxplots for Numerical Features:")
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_columns:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot for {col}")
    plt.show()

# Check skewness of numerical features
print("\nSkewness of Numerical Features:")
for col in numerical_columns:
    print(f"{col}: Skewness={skew(df[col].dropna())}, Kurtosis={kurtosis(df[col].dropna())}")

# Pairplot for visualizing feature interactions
print("\nPairplot for selected features:")
sns.pairplot(df[numerical_columns[:5]])  # Adjust [:5] to include relevant columns
plt.show()

# Value counts for categorical features
categorical_columns = df.select_dtypes(include=['object']).columns
for col in categorical_columns:
    print(f"\nValue Counts for {col}:")
    print(df[col].value_counts())

# Check distributions of numerical features
print("\nDistributions of Numerical Features:")
for col in numerical_columns:
    plt.figure()
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f"Distribution of {col}")
    plt.show()

# Check interactions of categorical features with target
if 'target' in df.columns:
    print("\nCategorical Features vs Target:")
    for col in categorical_columns:
        plt.figure()
        sns.countplot(x=col, hue='target', data=df)
        plt.title(f"{col} vs Target")
        plt.show()
